# 01 · Jakość pychameleona na datasetach z ground-truth

Cel: ocenić **ilościowo i jakościowo** jak nasza implementacja CHAMELEON-a radzi sobie na 4 syntetycznych datasetach z papera Karypis 1999 (DS1, DS3, DS4, DS5).

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import _helpers as H

plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 120
pd.options.display.float_format = '{:.3f}'.format


## 1. Tabela jakości — DSx datasety

Wybrane metryki na 4 datasetach Karypisa:

- **ARI** (external, chance-corrected) — pairwise agreement vs ground-truth. Standard literaturowy dla porównań z prawdą.
- **NMI** (external, entropy-based) — komplementarne do ARI; mierzy ile informacji o GT zawierają labelki.
- **Silhouette** (internal) — najczęstsza metryka jakości bez GT; uwaga: penalizuje non-convex shapes.
- **Quality score** — **(ARI + NMI) / 2**. Sensowna agregacja bo (a) obie metryki są w `[0,1]`, (b) obie są chance-corrected, (c) chwytają komplementarne aspekty (pairs vs entropia). Daje **jedną liczbę headline** którą można porównać między datasetami.

In [ ]:
comp = H.load_comparison().set_index('dataset')
dsx = comp.loc[H.DATASETS_WITH_GT, ['n', 'runtime_s', 'ari', 'nmi', 'silhouette',
                                     'homogeneity', 'completeness']].copy()
dsx['quality'] = (dsx['ari'] + dsx['nmi']) / 2
dsx = dsx[['n', 'runtime_s', 'ari', 'nmi', 'silhouette',
           'homogeneity', 'completeness', 'quality']]
dsx = dsx.sort_values('quality', ascending=False)
# Kolejność datasetów posortowana malejąco po Quality — używana też niżej w bar chart i galerii.
ORDER = list(dsx.index)
dsx_view = dsx.rename(columns={'runtime_s': 'runtime [s]', 'ari': 'ARI', 'nmi': 'NMI',
                                'silhouette': 'Silhouette', 'homogeneity': 'Homog.',
                                'completeness': 'Compl.', 'quality': 'Quality (ARI+NMI)/2'})
dsx_view.index = [H.PRETTY[d] for d in dsx_view.index]
dsx_view.style.format({
    'runtime [s]': '{:.2f}', 'ARI': '{:.3f}', 'NMI': '{:.3f}',
    'Silhouette': '{:.3f}', 'Homog.': '{:.3f}', 'Compl.': '{:.3f}',
    'Quality (ARI+NMI)/2': '{:.3f}',
}).background_gradient(subset=['Quality (ARI+NMI)/2'], cmap='RdYlGn', vmin=0, vmax=1)

**Czytanie tabeli:**

- **DS1: Quality = 1.000** — pychameleon idealnie rekonstruuje ground-truth.
- **DS5: 0.967, DS4: 0.905** — bardzo wysoka jakość mimo skomplikowanych kształtów (DS5 ma klastry różnych gęstości, DS4 ma 9 klastrów częściowo nakładających się).
- **DS3: 0.801** — najgorszy wynik, bottleneck to ~10% punktów oznaczonych w GT jako szum (`-1`). CHAMELEON przypisuje je do najbliższych klastrów (nie ma noise-handlingu), co psuje ARI.

## 2. Quality score per dataset — wykres słupkowy

In [ ]:
quality = (comp.loc[ORDER, 'ari'] + comp.loc[ORDER, 'nmi']) / 2
ari = comp.loc[ORDER, 'ari']
nmi = comp.loc[ORDER, 'nmi']

fig, ax = plt.subplots(figsize=(8, 3.8))
x = np.arange(len(ORDER))
w = 0.27
ax.bar(x - w, ari, w, label='ARI', color='#4C72B0')
ax.bar(x, nmi, w, label='NMI', color='#DD8452')
ax.bar(x + w, quality, w, label='Quality = (ARI+NMI)/2', color='#55A868')
ax.set_xticks(x)
ax.set_xticklabels([H.PRETTY[d] for d in ORDER])
ax.set_ylim(0, 1.05)
ax.axhline(1.0, ls=':', color='gray', linewidth=0.8)
ax.set_ylabel('score')
ax.set_title('Jakość pychameleona na datasetach Karypisa')
ax.legend(loc='lower left', framealpha=0.9)
for i, (a, n, q) in enumerate(zip(ari, nmi, quality)):
    ax.text(i - w, a + 0.015, f'{a:.2f}', ha='center', fontsize=8)
    ax.text(i, n + 0.015, f'{n:.2f}', ha='center', fontsize=8)
    ax.text(i + w, q + 0.015, f'{q:.2f}', ha='center', fontsize=8, fontweight='bold')
ax.grid(True, axis='y', linestyle=':', alpha=0.4)
plt.tight_layout()
plt.show()

## 3. Galeria 3-kolumnowa — ground truth · wynik · błędy

Datasety w tej samej kolejności co tabela (najlepszy Quality na górze, najgorszy na dole). Dla każdego:

1. **Ground truth** — prawdziwe klastry (`-1` szum szaro).
2. **pychameleon** — nasze labelki, po hungarian remappingu na przestrzeń GT (żeby kolory się zgadzały).
3. **Correctness** — zielony = poprawnie, czerwony = błąd, szary = szum GT (ignorowany w accuracy). Tytuł podaje **% poprawnie przypisanych punktów non-noise**.

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(11, 14))
for i, name in enumerate(ORDER):
    gt_df = H.load_ground_truth(name)
    pred_df = H.load_pychameleon_labels(name)
    pred_aligned = H.align_labels_to_gt(pred_df['label'].values, gt_df['label'].values)
    pred_view = pred_df.copy()
    pred_view['label'] = pred_aligned

    H.scatter(axes[i, 0], gt_df, title=f'{H.PRETTY[name]} — ground truth')
    H.scatter(axes[i, 1], pred_view, title=f'{H.PRETTY[name]} — pychameleon')
    nc, ni, nn = H.correctness_scatter(
        axes[i, 2], pred_df[['x0', 'x1']], gt_df['label'].values,
        pred_aligned, title='',
    )
    total_eval = nc + ni
    pct = 100 * nc / total_eval if total_eval else 0
    axes[i, 2].set_title(
        f'{H.PRETTY[name]} — accuracy {pct:.1f}%   ',
        fontsize=9,
    )
plt.tight_layout()
plt.show()

**Co wyciągamy z trzeciej kolumny:**

- **DS1**: 100% zielone — żadnego błędu, idealna rekonstrukcja.
- **DS5**: błędy skupione na styku 2 najbliższych klastrów — typowy artefakt mergera (niejasno gdzie kończy się jedna kropla a zaczyna druga).
- **DS4**: błędy na końcach wydłużonych kształtów + drobne mismatche między 9 klastrami.
- **DS3**: czerwień rozsiana po całym diagramie — to bezpośredni efekt punktów które w GT są szumem (`-1`), a pychameleon je przypisuje do najbliższego klastra; po hungarian matchingu trafiają w "nie ten klaster".

## 4. Wnioski

1. **Średnia jakość po 4 datasetach Karypisa = 0.93** (Quality score). Reprodukcja papera udana.
2. **DS1/DS5/DS4: > 95% punktów przypisanych do właściwego klastra**. Algorytm działa wybornie na kształtach dla których został zaprojektowany (non-convex, różne gęstości).
3. **DS3 to znana granica** — CHAMELEON w paperze nie ma noise-handlingu. Punkty oznaczone w GT jako szum trafiają do klastrów, co kosztuje ~20–30 punktów procentowych jakości. Workaround: post-processing oparty na density (np. odetnij punkty o niskim local connectivity) — to wykraczałoby poza scope odwzorowania papera.